# 🦙 Elyza-7B ファインチューニング & 評価ガイド

このノートブックは、Unslothを使用した高速ファインチューニングと、
体系的な推論テストを実行するための統合環境です。

---

## 📋 必要なファイル
1. **train_data_vX.jsonl** - 学習データセット
2. **config_vX.yaml** - ハイパーパラメータ設定
3. **train_vX.py** - 学習実行スクリプト

---

## ⚙️ Step 1: 環境セットアップ

Unslothライブラリをインストールし、GPU環境を初期化します。

In [ ]:
import torch

# GPU検出と互換性チェック
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU検出: {gpu_name}")
    print(f"📊 VRAM容量: {total_memory:.1f} GB")
    
    if "T4" not in gpu_name and "A100" not in gpu_name and "L4" not in gpu_name:
        print("⚠️ 警告: T4/L4/A100以外のGPUではUnslothの動作が保証されません")
else:
    raise RuntimeError("❌ GPUが検出されません\n→ [ランタイム] > [ランタイムのタイプを変更] > [T4 GPU] を選択してください")

# Unslothインストール
print("\n⏳ Unslothをインストール中... (2-3分かかります)")
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

print("✅ セットアップ完了")

---

## 💾 Step 2: Google Driveマウント

学習結果を永続化するためにDriveをマウントします。

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# 保存先ディレクトリの作成
DRIVE_SAVE_PATH = "/content/drive/MyDrive/Elyza_Models"
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

print(f"\n📂 保存先: {DRIVE_SAVE_PATH}")
print("✅ Driveマウント完了")

---

## 📤 Step 3: 学習ファイルのアップロード

以下の3ファイルをアップロードしてください：
- `train_data_vX.jsonl` (学習データ)
- `config_vX.yaml` (設定ファイル)
- `train_vX.py` (学習スクリプト)

In [ ]:
from google.colab import files
import os

print("📁 ファイル選択ダイアログを開いています...\n")
uploaded = files.upload()

# 必須ファイルの確認
required_files = ["config", "train", ".jsonl"]
status = {"config": False, "script": False, "dataset": False}

for filename in uploaded.keys():
    if "config" in filename and filename.endswith(".yaml"):
        status["config"] = True
        print(f"✅ 設定ファイル: {filename}")
    elif "train" in filename and filename.endswith(".py"):
        status["script"] = True
        print(f"✅ 学習スクリプト: {filename}")
    elif filename.endswith(".jsonl"):
        status["dataset"] = True
        print(f"✅ データセット: {filename}")

# 結果サマリー
print("\n" + "="*50)
if all(status.values()):
    print("🎉 すべての必須ファイルが揃いました")
else:
    print("⚠️ 不足しているファイル:")
    if not status["config"]: print("  - config_vX.yaml")
    if not status["script"]: print("  - train_vX.py")
    if not status["dataset"]: print("  - train_data_vX.jsonl")
print("="*50)

---

## 🚀 Step 4: 学習の実行

Unslothによる高速ファインチューニングを開始します。

⚠️ **注意**: 学習中はランタイムを切断しないでください

In [ ]:
import glob

# アップロードされたファイルから自動検出
config_file = glob.glob("config*.yaml")[0] if glob.glob("config*.yaml") else None
train_script = glob.glob("train*.py")[0] if glob.glob("train*.py") else None

if config_file and train_script:
    print(f"🎯 使用する設定: {config_file}")
    print(f"🎯 実行スクリプト: {train_script}\n")
    print("="*70)
    print("🚀 学習を開始します... (進捗はログで確認できます)")
    print("="*70 + "\n")
    
    !python {train_script} --config_path {config_file}
else:
    print("❌ エラー: 設定ファイルまたは学習スクリプトが見つかりません")
    print("→ Step 3を再実行してファイルをアップロードしてください")

---

## 🧪 Step 5: 推論テスト（体系的評価）

学習済みモデルを使用して、実務想定の質問セットで性能を評価します。

### 📊 評価カテゴリ
- **法的リスク検出** - 契約条項の問題点を特定できるか
- **条項整合性チェック** - 矛盾する規定を発見できるか
- **法改正対応** - 最新法令への適合性を判断できるか

In [ ]:
from unsloth import FastLanguageModel
from transformers import logging as transformers_logging
import torch
import os

# 警告を抑制
transformers_logging.set_verbosity_error()

# === 設定 ===
ADAPTER_PATH = "./results_unsloth/final_adapter"
MAX_SEQ_LENGTH = 2048
# ============

if not os.path.exists(ADAPTER_PATH):
    raise FileNotFoundError(
        f"❌ モデルが見つかりません: {ADAPTER_PATH}\n"
        "→ Step 4の学習が完了しているか確認してください"
    )

print("📦 モデルをロード中...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = ADAPTER_PATH,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True,
    fix_tokenizer = True,
    ignore_mismatched_sizes = True,
)
FastLanguageModel.for_inference(model)
print("✅ モデルロード完了\n")

# === 推論関数 ===
def ask_legal_question(instruction, input_text=""):
    """法務特化の質問応答関数"""
    system_prompt = (
        "あなたはIT法務およびコンプライアンスの専門コンサルタントです。\n"
        "正確な法的根拠に基づき、実務的な改善提案を行ってください。"
    )
    
    prompt_template = (
        f"<s>[INST] <<SYS>>\n{system_prompt}\n<</SYS>>\n\n"
        f"{instruction}\n\n{input_text} [/INST]"
    )
    
    inputs = tokenizer(
        [prompt_template],
        return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens = 512,
        use_cache = True,
        temperature = 0.6,
        top_p = 0.9,
        repetition_penalty = 1.2
    )
    
    result = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    return result.split("[/INST]")[-1].strip()

print("🔧 推論エンジン準備完了")

### 📝 評価用質問セット

以下のセルで質問を定義し、実行することで体系的な評価が可能です。

**質問の構造**:
- `title`: 評価カテゴリ（ログ出力用）
- `content`: 実際の質問文（条項・事例を含む）

In [ ]:
# === 評価用質問セット ===
# ※ バージョンごとに質問内容を変更してテストしてください

test_questions = [
    {
        "title": "消費者契約法違反の検出",
        "content": """以下の利用規約の条項について、日本の消費者契約法の観点から問題となる可能性のある箇所を指摘し、その理由と改善案を提示してください。

『第8条（免責事項）
当社は、本サービスの提供に関して、いかなる場合においても一切の責任を負わないものとします。また、ユーザーが本サービスの利用により被った損害について、その原因の如何を問わず、当社は賠償責任を負いません。』"""
    },
    {
        "title": "契約条項の矛盾分析",
        "content": """以下の2つの条項間に矛盾や整合性の問題がないか分析してください。

『第3条（契約期間）
本契約の有効期間は1年間とし、期間満了の1ヶ月前までに書面による解約通知がない場合、自動的に1年間更新されるものとします。』

『第12条（解約）
ユーザーは、いつでも自由に本サービスを解約することができます。解約は、アプリ内の設定画面から即座に行うことができます。』"""
    },
    {
        "title": "法改正対応の必要性判断",
        "content": """以下の個人情報取扱条項について、改正個人情報保護法（2022年4月施行）および電気通信事業法改正（2023年6月施行）を踏まえ、更新が必要な箇所を指摘してください。

『第7条（個人情報の取扱い）
当社は、ユーザーの個人情報を適切に管理し、第三者に提供する場合は事前に通知します。また、Cookie等の情報も収集し、サービス改善に利用します。』"""
    }
]

print(f"📋 評価質問数: {len(test_questions)}件")
print("✅ 質問セット準備完了")

### ▶️ 推論実行

定義した質問セットに対してモデルの回答を生成します。

In [ ]:
# === 推論実行 ===
print("\n" + "="*70)
print("🧪 推論テスト開始")
print("="*70 + "\n")

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*70}")
    print(f"📌 質問 {i}: {question['title']}")
    print(f"{'='*70}")
    print(f"\n【質問内容】\n{question['content']}")
    print(f"\n{'─'*70}")
    
    # 回答生成
    answer = ask_legal_question(question['content'])
    
    print(f"【モデル回答】\n{answer}")
    print(f"\n{'='*70}\n")

print("\n✅ 全ての推論テストが完了しました")

---

## 💾 Step 6: モデルの保存

学習済みアダプターをGoogle Driveにバックアップします。

In [ ]:
import shutil
import datetime

SOURCE_DIR = "./results_unsloth/final_adapter"
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
version = "v7"  # ← バージョン番号を手動で変更

DEST_DIR = f"{DRIVE_SAVE_PATH}/Elyza-7B-{version}_{timestamp}"

if os.path.exists(SOURCE_DIR):
    print(f"⏳ モデルをコピー中...")
    
    # 既存ディレクトリがあれば削除
    if os.path.exists(DEST_DIR):
        shutil.rmtree(DEST_DIR)
    
    shutil.copytree(SOURCE_DIR, DEST_DIR)
    
    # 保存内容の確認
    saved_files = os.listdir(DEST_DIR)
    total_size = sum(
        os.path.getsize(os.path.join(DEST_DIR, f)) 
        for f in saved_files
    ) / (1024**2)  # MB単位
    
    print("\n" + "="*70)
    print("✅ Google Driveへの保存が完了しました")
    print("="*70)
    print(f"📂 保存先: {DEST_DIR}")
    print(f"📊 ファイル数: {len(saved_files)}個")
    print(f"💾 合計サイズ: {total_size:.1f} MB")
    print("="*70)
else:
    print("❌ 保存元のモデルが見つかりません")
    print(f"→ 確認パス: {SOURCE_DIR}")

---

## 📊 学習ログの確認（オプション）

学習中のLoss推移を可視化します。

In [ ]:
import json
import matplotlib.pyplot as plt

LOG_FILE = "./results_unsloth/trainer_state.json"

if os.path.exists(LOG_FILE):
    with open(LOG_FILE, 'r') as f:
        trainer_state = json.load(f)
    
    # Loss履歴の抽出
    log_history = trainer_state.get('log_history', [])
    steps = [entry['step'] for entry in log_history if 'loss' in entry]
    losses = [entry['loss'] for entry in log_history if 'loss' in entry]
    
    # グラフ描画
    plt.figure(figsize=(10, 5))
    plt.plot(steps, losses, marker='o', linewidth=2, markersize=4)
    plt.title('Training Loss Progress', fontsize=14, fontweight='bold')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"\n📉 最終Loss: {losses[-1]:.4f}")
    print(f"📉 最小Loss: {min(losses):.4f} (Step {steps[losses.index(min(losses))]})")
else:
    print("⚠️ ログファイルが見つかりません")

---

## 🎯 次のステップ

### 評価のポイント
1. **法的根拠の正確性** - 条文番号や法律名が正しいか
2. **質問の意図理解** - 聞かれたことに正面から答えているか
3. **ハルシネーション** - 存在しない情報を創作していないか
4. **実務的有用性** - 提案が現実的で実装可能か

### 改善サイクル
1. 推論結果を分析 → 問題点を特定
2. 学習データまたはハイパーパラメータを調整
3. 再学習 → 再評価

---

**📌 注意事項**
- 学習中はランタイムを切断しないでください
- 90分ごとにセッションがリセットされる可能性があります
- 重要なモデルは必ずGoogle Driveに保存してください

---

*最終更新: 2025年12月23日*